# 03. Mô hình và đánh giá

**Mục tiêu:** so sánh 4 mô hình dự báo ngắn hạn bằng đánh giá **rolling-origin** ở tầm 1, 3 và 6 tháng (Chương 5 của báo cáo).

**4 mô hình (mỗi cái một câu):**
- **Naive:** tháng sau bằng tháng này.
- **Seasonal Naive:** tháng sau bằng đúng tháng đó của năm ngoái.
- **Trung bình trượt (k=3):** tháng sau bằng trung bình 3 tháng gần nhất.
- **Hồi quy xu hướng (OLS):** kẻ một đường thẳng qua dữ liệu rồi kéo dài ra.

**Rolling-origin:** đứng ở từng tháng trong quá khứ (từ tháng thứ 60 trở đi), chỉ dùng dữ liệu *trước* tháng đó để dự báo h tháng sau, rồi so với giá trị thật và lấy trung bình sai số. Mô hình không bao giờ được nhìn trước tương lai.

**Thước đo:** MAE (sai số trung bình, cùng đơn vị với chuỗi) và MAPE (sai số trung bình theo %). Càng nhỏ càng tốt.

In [1]:
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## 1. Nạp dữ liệu, khai báo mô hình và hàm chạy

In [2]:
from cpi_forecast.evaluation.rolling_origin import rolling_origin_eval

df = pd.read_csv("../data/processed/cpi_monthly_clean.csv", parse_dates=["date"])

from cpi_forecast.models.naive import NaiveForecaster
from cpi_forecast.models.seasonal_naive import SeasonalNaiveForecaster
from cpi_forecast.models.moving_average import MovingAverageForecaster
from cpi_forecast.models.trend_ols import TrendOLSForecaster

models = {
    "Naive": NaiveForecaster,
    "Seasonal Naive": SeasonalNaiveForecaster,
    "Trung bình trượt (k=3)": lambda: MovingAverageForecaster(k=3),
    "Hồi quy xu hướng (OLS)": TrendOLSForecaster,
}


def run_all(y):
    rows = []
    for name, factory in models.items():
        res = rolling_origin_eval(y, factory, horizons=(1, 3, 6), min_train=60)
        row = {"Mô hình": name}
        for h, (mae_v, mape_v) in res.items():
            row[f"MAE h={h}"] = round(mae_v, 3)
            row[f"MAPE h={h}"] = round(mape_v, 3)
        rows.append(row)
    return pd.DataFrame(rows)


## 2. Kết quả trên chuỗi chỉ số (`index`)

In [3]:
table_index = run_all(df["index"].to_numpy())
table_index

,Mô hình,MAE h=1,MAPE h=1,MAE h=3,MAPE h=3,MAE h=6,MAPE h=6
0,Naive,1.114,0.451,2.895,1.167,5.047,2.008
1,Seasonal Naive,10.493,4.324,10.403,4.254,10.134,4.087
2,Trung bình trượt (k=3),1.985,0.811,3.633,1.474,5.849,2.335
3,Hồi quy xu hướng (OLS),13.791,5.349,14.660,5.690,15.886,6.151


In [4]:
# Sai số của mỗi mô hình chia cho sai số của Naive (Naive = 1,00)
cot = ["MAE h=1", "MAE h=3", "MAE h=6"]
t = table_index.set_index("Mô hình")[cot]
(t / t.loc["Naive"]).round(2)

,MAE h=1,MAE h=3,MAE h=6
Mô hình,,,
Naive,1.00,1.00,1.00
Seasonal Naive,9.42,3.59,2.01
Trung bình trượt (k=3),1.78,1.25,1.16
Hồi quy xu hướng (OLS),12.38,5.06,3.15


**Cách đọc:** số lớn hơn 1 nghĩa là sai nhiều hơn Naive bấy nhiêu lần.

- **Naive có sai số nhỏ nhất ở cả ba tầm.** Sai số của Naive và trung bình trượt tăng dần theo tầm dự báo, vì chỉ số tăng liên tục nên dự báo "không đổi" càng xa càng lệch.
- **Seasonal Naive** có sai số gần như không đổi (khoảng 10 điểm): nó luôn chậm hơn thực tế khoảng một năm tăng trưởng.
- **OLS** sai nhiều nhất: một đường thẳng không khớp với đường cong của chỉ số.
- MAPE đều nhỏ vì chỉ số đã lớn hơn 100; vì vậy MAE cho thấy khác biệt rõ hơn.

## 3. Kết quả trên chuỗi % thay đổi (`pct`)

**Lưu ý:** MAPE ra `inf` ở chuỗi này **không phải lỗi**. Có tháng `pct` đúng bằng 0, chia cho 0 ra vô cực. Ta chỉ đọc **MAE**. Cũng đừng so MAE của chuỗi này với chuỗi chỉ số vì khác đơn vị.

In [5]:
table_pct = run_all(df["pct"].to_numpy())
table_pct

/home/claude/work/notebooks/../src/cpi_forecast/evaluation/metrics.py:7: RuntimeWarning: divide by zero encountered in divide
  return 100 * np.mean(np.abs((y_true - y_pred) / y_true))


,Mô hình,MAE h=1,MAPE h=1,MAE h=3,MAPE h=3,MAE h=6,MAPE h=6
0,Naive,0.375,inf,0.526,inf,0.534,inf
1,Seasonal Naive,0.517,inf,0.520,inf,0.493,inf
2,Trung bình trượt (k=3),0.406,inf,0.451,inf,0.453,inf
3,Hồi quy xu hướng (OLS),0.418,inf,0.424,inf,0.418,inf


**Cách đọc:** thứ hạng **đảo ngược** so với chuỗi chỉ số.

- Tầm 1 tháng: Naive tốt nhất.
- Tầm 3 và 6 tháng: Naive kém nhất, còn OLS tốt nhất.

Lý do khả dĩ (chưa kiểm chứng riêng): % thay đổi mỗi tháng nhảy lên xuống khá ngẫu nhiên, nên dùng đúng con số của tháng vừa rồi làm dự báo xa là bám theo một con số nhiễu; đường OLS trên chuỗi này gần như nằm ngang quanh mức trung bình nên ổn định hơn khi nhìn xa. Chênh lệch giữa các mô hình ở chuỗi này khá nhỏ, nên không kết luận quá mạnh.

**Kết luận chung:** mô hình tốt nhất phụ thuộc vào chuỗi được dự báo và tầm dự báo.

## 4. Lưu kết quả

In [6]:
import os

os.makedirs("../results/tables", exist_ok=True)
table_index.to_csv("../results/tables/rolling_origin_index.csv", index=False)
table_pct.to_csv("../results/tables/rolling_origin_pct.csv", index=False)
print("Đã lưu 2 file vào results/tables/")

Đã lưu 2 file vào results/tables/
